# AccessGuard — Notebook 02: Dataset Normalization & Split Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

Normalizes source dataset labels into AccessGuard's 6 standard categories:
- `curb_ramp`
- `stairs`
- `surface_damage`
- `obstacle`
- `crosswalk`
- `unknown_infrastructure`

**Critical Rule:** Roadway cracks are strictly tagged as `unknown_infrastructure` unless verified on pedestrian footpaths.


In [ ]:
!pip install -q pyyaml pillow numpy


In [ ]:
# AccessGuard Taxonomy Definition
ACCESSGUARD_CLASSES = [
    'curb_ramp',
    'stairs',
    'surface_damage',
    'obstacle',
    'crosswalk',
    'unknown_infrastructure'
]
CLASS_TO_IDX = {name: i for i, name in enumerate(ACCESSGUARD_CLASSES)}
print('AccessGuard Classes:', CLASS_TO_IDX)


In [ ]:
# Semantic Mapping Function
def map_label(source_dataset, source_label, is_sidewalk=False):
    lbl = str(source_label).lower().strip()
    if 'ramp' in lbl:
        return 'curb_ramp'
    elif 'stair' in lbl or 'step' in lbl:
        return 'stairs'
    elif 'damage' in lbl or 'pothole' in lbl:
        return 'surface_damage' if is_sidewalk else 'unknown_infrastructure'
    elif 'obstacle' in lbl or 'pole' in lbl or 'bollard' in lbl:
        return 'obstacle'
    elif 'crosswalk' in lbl or 'crossing' in lbl:
        return 'crosswalk'
    return 'unknown_infrastructure'

print('Test Road Crack:', map_label('SVRDD', 'D20_alligator_crack', is_sidewalk=False))
print('Test Sidewalk Crack:', map_label('SVRDD', 'D20_alligator_crack', is_sidewalk=True))
print('Test Ramp:', map_label('ProjectSidewalk', 'CurbRamp'))


In [ ]:
# Generate dataset.yaml for YOLO training
import yaml
import os

os.makedirs('dataset', exist_ok=True)
yolo_yaml = {
    'path': os.path.abspath('dataset'),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {i: name for i, name in enumerate(ACCESSGUARD_CLASSES)}
}
with open('dataset/accessguard.yaml', 'w') as f:
    yaml.dump(yolo_yaml, f, sort_keys=False)
print('Generated dataset/accessguard.yaml successfully!')
